## Data Wrangling

Recall our data wrangling process. The slides and this week's chapter detail the process.  

1. Discovering
2. Structuring
3. Cleaning 
4. Enriching
5. Validating
6. Publishing

We'll explore the process and the various techniques that can be used in Python to address the steps.

IMPORTANT NOTE: USE YOUR NOTEBOOK TO KEEP TRACK OF HOW YOU CLEANED DATA. NEVER CHANGE DATA AND THEN REMOVE THE CODE THAT CHANGED THE DATA. 

### 1. Discovering

We have found data concerning college student food preferences. This data is the result of a student survey at Mercyhurst University. The file is: food_coded.csv

Data is from https://www.kaggle.com/datasets/borapajo/food-choices/data. About the data: "This dataset includes information on food choices, nutrition, preferences, childhood favorites, and other information from college students. There are 126 responses from students. Data is raw and uncleaned." Warning: there is some profanity. 

Let's do our import and load this data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load the dataset
df = pd.read_csv("food_coded.csv")

In [ ]:
# Do our typical review of the data to get started
# see first five and last five observations
df

In [ ]:
# Note that this does not show us all columns
pd.pandas.set_option('display.max_columns', None)

In [ ]:
# See basic info - columns, non-null count, and datatype
df.info()

In [ ]:
# See/describe basic statistics
df.describe()

### 2. Structuring

Next we'll structure the data. We want to standardize the format and fix any formatting issues.  Are the units okay? Do we have any date/time issues? Are there case issues? ...

Helpful functions here:
* value_counts() for features that are categorical, we can see the unique categories and their counts https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.value_counts.html
* astype() for converting column to correct data type https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html
* fillna() for replacing NaNs with a number https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html
* string manipulation functions (e.g. replace(), strip(), lower(), ...) https://www.w3schools.com/python/python_ref_string.asp
* and more

Example 1: One item to fix is employment. It should be 1, 2, 3, or 4 and we have instead floating-point numbers.  
Consider what type of data this really is? Categorical? Numerical? Nominal? Ordinal?

Why convert to a category if categorical? 
1. It saves memory space.
2. You can define a custom sort order for categorical data (e.g. XS, M, L, XL).
3. Faster calculation speed.

See [https://pandas.pydata.org/docs/user_guide/categorical.html](https://pandas.pydata.org/docs/user_guide/categorical.html) for more info on categories.

In [ ]:
# Let's see what we have and then fix the data type (count the values)
print(df["employment"].value_counts())

#df["employment"] = df["employment"].astype(int) - cannot convert nulls
# we have nulls - let's use 0 to represent no answer -- need to fill these values before moving on
df["employment"] = df["employment"].fillna(0)
df["employment"] = df["employment"].astype("int")

# convert this to "category" type
# df["employment"] = df["employment"].astype("category")
# look at describe at the top and now it will show employment as a series

# Let's see what we have and then fix the data type (count the values)
print(df["employment"].value_counts())

In [ ]:
# check it out now - see info()
df.info()

Example 2: Favorite cuisine looks messy. Let's ensure all the field values are in lowercase.

In [ ]:
# See the problem first (count the values)
#df["fav_cuisine"].value_counts()
# Fix the string case
df["fav_cuisine"] = df["fav_cuisine"].str.lower()
# remove word "food" (replace) - we already forced this to lowercase
df["fav_cuisine"] = df["fav_cuisine"].str.replace("food|cuisine|\\.", "", regex=True)
# remove/strip extra spaces
df["fav_cuisine"] = df["fav_cuisine"].str.strip()
# See fixed
df["fav_cuisine"].value_counts()

There is much, much more we can do to structure the data, and we'll do a little more of this next week. But let's look at the next step for now.

### 3. Data Cleaning

We look for dirty data (e.g. duplicates, missing data, outliers, duplicates).  There are many functions what will help us here, such as:
* duplicated() to determine which records/observations/instances are duplicates
* drop_duplicates() to drop duplicate records
* isnull() to detect missing values - returns boolean values
* dropna() to drop rows with missing values
* drop() to drop a feature/column

Let's begin by finding any rows that are duplicates of other rows. If we find any duplicated rows, let's remove them.

In [ ]:
# check for duplicates
df.duplicated()

In [ ]:
# Let's see which records are duplicates
df[df.duplicated()]

In [ ]:
# Let's drop the duplicate
df = df.drop_duplicates()

Next we'll consider if data is missing. Let's review what is missing first and then decide next steps.

In [ ]:
# Let's see what fields are missing data
missing = df.isnull().sum().sort_values()

In [ ]:
# cheating - convert series to a string to print it all
df.isnull().sum().sort_values().to_string()

In [ ]:
missing[missing > 0]

If all the values are missing, we really should drop the row/observation.

In [ ]:
# drop rows where all (how) data is na
df = df.dropna(how="all")

If a row is missing a significant number of values, we can then drop that row.

In [ ]:
# Let's drop rows missing more than 5 values (threshold)
df = df.dropna(thresh=len(df.columns)-5)

We need to consider what percentage of values are missing from a feature and if it is random or not. Depending on how much data is missing, we'll take different actions. 

Let's check out calories_day. Which rows are missing values? Is there a certain population that does not provide this data? Is the data missing at random, or is there a pattern? What percentage of the values are missing?

In [ ]:
# which fields are missing calories - let's explore this data (filter by is null)
df[df["calories_day"].isnull()]

In [ ]:
# Let's look at calories_day - what percentage of records is missing?
df["calories_day"].isnull().sum()/len(df)

In [ ]:
# for 15%, let's fill the missing values (na) with the mean or median
# check if we want mean or median
print(df["calories_day"].value_counts())

# get mean or median
new_value = df["calories_day"].value_counts().idxmax()
print("New value is", new_value)

# replace missing values with mean or median
df["calories_day"] = df["calories_day"].fillna(new_value)
print(df["calories_day"].value_counts())

If more than 50% of the values of a feature are missing, we'd most likely drop the feature. 

In [ ]:
# if type_sports was a high percentage (let's say 60%) we should just remove/drop the feature


To be continued next class